# A Pearl episode beside a DROID episode

This notebook opens one packaged Pearl episode and DROID's LeRobot copy side by side: the stream facts, the
columns, and one frame from each. It is the demo for DROID campuses (`.claude/PLAN.md` section 7).

**Before you run it**

```bash
uv sync --extra datasets
uv run python scripts/make_synthetic_take.py demo      # a synthetic take, until you have a real one
uv run jupyter lab notebooks/pearl_beside_droid.ipynb
```

Point `PEARL_LEROBOT` at the `lerobot/` folder of a real packaged episode to use your own take. The synthetic
take is an ffmpeg test pattern, not Pearl footage. Outputs stay cleared in git because frames can show people.

**Sources**

- DROID copy: `lerobot/droid_1.0.1` on the Hugging Face Hub, `meta/info.json` (`codebase_version v3.0`).
- LeRobot format and merge rule: https://huggingface.co/docs/lerobot/main/en/video_encoding_parameters
- DROID dataset: https://droid-dataset.github.io/droid/the-droid-dataset.html

In [ ]:
import io
import json
import os
from pathlib import Path

import av
import httpx
import matplotlib.pyplot as plt
import pandas as pd
from huggingface_hub import hf_hub_download
from lerobot.datasets import LeRobotDataset

PEARL_LEROBOT = Path(os.environ.get("PEARL_LEROBOT", "demo/episodes/synthetic-0001/lerobot"))
DROID_REPO = "lerobot/droid_1.0.1"
# Only the first few megabytes of one DROID video shard: a full episode is about 1.7 GB across three cameras.
DROID_SHARD = f"https://huggingface.co/datasets/{DROID_REPO}/resolve/main/videos/observation.images.exterior_1_left/chunk-000/file-000.mp4"
assert PEARL_LEROBOT.exists(), f"{PEARL_LEROBOT} not found; see the first cell"

## Load both

The Pearl episode opens with LeRobot's own loader (`pyav` backend, so it works with any installed FFmpeg).
For DROID we read the small `info.json`, and decode a few frames from the start of one video shard.

In [ ]:
pearl = LeRobotDataset("pearl/episode", root=PEARL_LEROBOT, video_backend="pyav")
pearl_info = json.loads((PEARL_LEROBOT / "meta" / "info.json").read_text())
droid_info = json.loads(Path(hf_hub_download(DROID_REPO, "meta/info.json", repo_type="dataset")).read_text())
print(f"Pearl: {len(pearl)} frames, {pearl.meta.total_episodes} episode(s), fps {pearl.fps}")
print(
    f"DROID: {droid_info['total_episodes']:,} episodes, {droid_info['total_frames']:,} frames, "
    f"fps {droid_info['fps']}"
)

## The stream facts, side by side

Every value comes from each dataset's own `meta/info.json`.

In [ ]:
def video_rows(info, label):
    rows = []
    for key, feature in info["features"].items():
        if feature.get("dtype") == "video":
            facts = feature["info"]
            rows.append(
                {
                    "dataset": label,
                    "camera key": key,
                    "codec": facts.get("video.codec"),
                    "pixel format": facts.get("video.pix_fmt"),
                    "size (w x h)": f"{facts.get('video.width')} x {facts.get('video.height')}",
                    "fps": facts.get("video.fps"),
                }
            )
    return rows


pd.DataFrame(video_rows(pearl_info, "Pearl") + video_rows(droid_info, "DROID (LeRobot copy)"))

In [ ]:
def other_columns(info):
    return sorted(k for k, v in info["features"].items() if v.get("dtype") != "video")


print("Pearl columns: ", other_columns(pearl_info))
print()
print("DROID columns: ", other_columns(droid_info))
print()
print("Pearl records video only: robot state and actions stay with the lab's own data.")

## One frame from each

Pearl: the first and last frame of the episode. DROID: the first frame of the first shard of `exterior_1_left`.

In [ ]:
def droid_first_frame():
    head = httpx.get(DROID_SHARD, headers={"Range": "bytes=0-6291455"}, follow_redirects=True, timeout=120)
    head.raise_for_status()
    with av.open(io.BytesIO(head.content)) as container:
        return next(container.decode(container.streams.video[0])).to_ndarray(format="rgb24")


camera = pearl.meta.video_keys[0]
shots = [
    (f"Pearl, {camera}, frame 0", pearl[0][camera].permute(1, 2, 0).numpy()),
    (f"Pearl, {camera}, last frame", pearl[len(pearl) - 1][camera].permute(1, 2, 0).numpy()),
    ("DROID, exterior_1_left, first frame of shard", droid_first_frame()),
]
fig, axes = plt.subplots(1, len(shots), figsize=(5 * len(shots), 3.2))
for ax, (title, image) in zip(axes, shots, strict=True):
    ax.imshow(image)
    ax.set_title(f"{title}\n{image.shape[1]} x {image.shape[0]}", fontsize=9)
    ax.axis("off")
plt.tight_layout()

## What this shows

- Both are LeRobot v3.0 with H.264 or AV1 in MP4, `yuv420p`. Pearl's file is the original H.264, copied as it is.
- DROID's copy is AV1 at 320 x 180 and 15 fps. The Pearl episode keeps the resolution and frame rate it was
  recorded at.
- LeRobot's `merge_datasets` requires `video.codec`, `video.pix_fmt`, `video.height`, `video.width` and
  `video.fps` to match across sources. So a Pearl episode sits **beside** DROID, as a sibling dataset. Putting
  it inside DROID's format would mean re-encoding and resampling. Bench issue #11 measures what that costs.
- The step table is indexed by frame number. This notebook makes no claim about timing between cameras.